# Fine-tune PhoBERT 5 lớp cảm xúc tiếng Việt

Notebook này bám theo `PhoBERT_5Class_Vietnamese_Sentiment_Pipeline.md`: chỉ dùng `text` làm input; `label` 1–5 là target. Không đưa `rating_raw`, star, ID hay metadata vào model. Train tối đa 50 epoch, chọn best checkpoint theo validation macro-F1 và early stopping.


In [1]:
from pathlib import Path
import json, random, re, unicodedata
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score, cohen_kappa_score, mean_absolute_error
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'phanloaicamxuc'
ARTIFACT_DIR = ROOT / 'selector' / 'artifacts' / 'phobert_5class'
MODEL_NAME = 'vinai/phobert-base-v2'
MAX_LENGTH = 160
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(DATA_DIR, ARTIFACT_DIR)


C:\Users\admin\OneDrive - The University of Technology\Desktop\pbl4-edge-model-distribution\data\phanloaicamxuc C:\Users\admin\OneDrive - The University of Technology\Desktop\pbl4-edge-model-distribution\selector\artifacts\phobert_5class


In [2]:
def normalize_text(value):
    text = unicodedata.normalize('NFC', str(value)).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def load_split(name):
    frame = pd.read_csv(DATA_DIR / f'{name}.csv')
    required = {'text', 'label'}
    if not required.issubset(frame.columns):
        raise ValueError(f'{name}.csv must contain text,label')
    frame = frame[['text', 'label']].dropna().copy()
    frame['text'] = frame['text'].map(normalize_text)
    frame['label'] = pd.to_numeric(frame['label'], errors='raise').astype(int)
    frame = frame[frame['text'].ne('') & frame['label'].between(1, 5)].reset_index(drop=True)
    return frame

train_df = load_split('train')
val_df = load_split('val')
print('train:', train_df.shape, train_df['label'].value_counts().sort_index().to_dict())
print('val:', val_df.shape, val_df['label'].value_counts().sort_index().to_dict())


train: (4220, 2) {1: 872, 2: 807, 3: 711, 4: 671, 5: 1159}
val: (1056, 2) {1: 218, 2: 202, 3: 178, 4: 168, 5: 290}


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.encodings = tokenizer(frame['text'].tolist(), truncation=True, max_length=MAX_LENGTH)
        self.labels = (frame['label'].to_numpy() - 1).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, index):
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

train_dataset = SentimentDataset(train_df)
val_dataset = SentimentDataset(val_df)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=5,
    id2label={i: f'LABEL_{i+1}' for i in range(5)},
    label2id={f'LABEL_{i+1}': i for i in range(5)},
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predicted = np.argmax(logits, axis=-1) + 1
    actual = labels + 1
    return {
        'accuracy': float(accuracy_score(actual, predicted)),
        'macro_f1': float(f1_score(actual, predicted, average='macro', zero_division=0)),
        'mae': float(mean_absolute_error(actual, predicted)),
        'qwk': float(cohen_kappa_score(actual, predicted, weights='quadratic')),
    }

args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / 'checkpoints'),
    learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
    save_total_limit=2, fp16=torch.cuda.is_available(), logging_steps=50, report_to='none', seed=SEED,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_dataset, eval_dataset=val_dataset,
    tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
trainer.train()


C:\Users\admin\AppData\Local\Temp\ipykernel_26852\1567478516.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Mae,Qwk
1,0.760800,0.654427,0.752841,0.735187,0.272727,0.926222
2,0.516600,0.564915,0.788826,0.768914,0.249053,0.925081
3,0.394000,0.537868,0.819129,0.807307,0.217803,0.926611
4,0.288700,0.559382,0.823864,0.811005,0.203598,0.938797
5,0.197300,0.594801,0.843750,0.834944,0.186553,0.938486
6,0.094300,0.683587,0.851326,0.843497,0.174242,0.943975
7,0.121800,0.771798,0.840909,0.830826,0.193182,0.934976
8,0.090100,0.705949,0.855114,0.847767,0.170455,0.945126
9,0.085600,0.841781,0.856061,0.849050,0.167614,0.946146
10,0.098800,0.833790,0.843750,0.838348,0.180871,0.944354


TrainOutput(global_step=3696, training_loss=0.23425093060964114, metrics={'train_runtime': 874.4781, 'train_samples_per_second': 241.287, 'train_steps_per_second': 15.095, 'total_flos': 1462511360728824.0, 'train_loss': 0.23425093060964114, 'epoch': 14.0})

In [5]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
metrics = trainer.evaluate(val_dataset)
trainer.save_model(ARTIFACT_DIR)
tokenizer.save_pretrained(ARTIFACT_DIR)
(ARTIFACT_DIR / 'training_config.json').write_text(json.dumps({
    'model_name': MODEL_NAME, 'max_length': MAX_LENGTH, 'epochs': EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'seed': SEED,
    'labels': [1, 2, 3, 4, 5], 'validation_metrics': metrics,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))


{
  "eval_loss": 0.7560425400733948,
  "eval_accuracy": 0.8636363636363636,
  "eval_macro_f1": 0.8586886321535205,
  "eval_mae": 0.16287878787878787,
  "eval_qwk": 0.9462871330407567,
  "eval_runtime": 2.2819,
  "eval_samples_per_second": 462.781,
  "eval_steps_per_second": 28.924,
  "epoch": 14.0
}


In [6]:
def classify_comment(comment):
    inputs = tokenizer([normalize_text(comment)], return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    device = next(trainer.model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        probabilities = torch.softmax(trainer.model(**inputs).logits, dim=-1)[0].cpu().numpy()
    label = int(np.argmax(probabilities) + 1)
    return {'score': label, 'probabilities': probabilities.round(4).tolist(), 'confidence': float(probabilities.max())}

for comment in ['Ứng dụng chạy rất nhanh và ổn định', 'Tải model quá lâu, máy bị giật lag']:
    print(comment, '=>', classify_comment(comment))


Ứng dụng chạy rất nhanh và ổn định => {'score': 5, 'probabilities': [9.999999747378752e-05, 9.999999747378752e-05, 9.999999747378752e-05, 0.0012000000569969416, 0.9983999729156494], 'confidence': 0.9984230995178223}
Tải model quá lâu, máy bị giật lag => {'score': 1, 'probabilities': [0.9987999796867371, 0.0005000000237487257, 0.00019999999494757503, 0.00019999999494757503, 0.0003000000142492354], 'confidence': 0.9987891316413879}
